# Airbnb Thessaloniki Listing 

## Building a predictive model to forecast pricing 

## Import libraries

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns 
import numpy as np
# import tensorflow as tf
# from tensorflow.keras.models import Sequential
# from tensorflow.keras.layers import Dense
from sklearn.model_selection import train_test_split
# from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
# from xgboost import XGBRegressor

## Data loading

In [2]:
# listings = pd.read_csv('thessaloniki_listings.csv')
listings = pd.read_csv('listings.csv')
# reviews = pd.read_csv('reviews.csv')

## Data cleaning

### Checking for missing values

In [3]:
print(listings.isnull().sum())

# Fill missing values (if applicable)
listings['name'].fillna('No Name', inplace=True)
listings['reviews_per_month'].fillna(0, inplace=True)

id                                   0
name                                 0
host_id                              0
host_name                          202
neighbourhood_group               4932
neighbourhood                        0
latitude                             0
longitude                            0
room_type                            0
price                              325
minimum_nights                       0
number_of_reviews                    0
last_review                        638
reviews_per_month                  638
calculated_host_listings_count       0
availability_365                     0
number_of_reviews_ltm                0
license                             31
dtype: int64


C:\Users\κική\AppData\Local\Temp\ipykernel_12160\2751709751.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  listings['name'].fillna('No Name', inplace=True)
C:\Users\κική\AppData\Local\Temp\ipykernel_12160\2751709751.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For examp

### Handling Duplicates

In [4]:
listings.drop_duplicates(inplace=True)

### Fixing Data Types

In [5]:
listings['last_review'] = pd.to_datetime(listings['last_review'], errors='coerce')

## Overall Summary

### How many listings are there?

In [6]:
print(f"Total listings: {listings.shape[0]}")

Total listings: 4932


In [7]:
listings = listings.dropna(subset=['last_review'])

## Defining the target and features

In [8]:
# Drop rows with missing prices
listings = listings.dropna(subset=['price']) 
listings['price'] = listings['price'].astype(float)

In [9]:
# Dropping columns
columns_to_drop = ['id', 'name', 'host_id', 'host_name', 'neighbourhood_group','license', 'last_review']
data_cleaned = listings.drop(columns=columns_to_drop)

In [10]:
# Interpolating 'lat' and 'long' based on 'neighbourhood'
# Grouping by 'neighbourhood' and calculating the mean 'lat' and 'long'
mean_coords = data_cleaned.groupby('neighbourhood')[['latitude', 'longitude']].mean()

# Applying the mean coordinates to missing values
data_cleaned = data_cleaned.set_index('neighbourhood')
data_cleaned['latitude'].fillna(mean_coords['latitude'], inplace=True)
data_cleaned['longitude'].fillna(mean_coords['longitude'], inplace=True)
data_cleaned.reset_index(inplace=True)

C:\Users\κική\AppData\Local\Temp\ipykernel_12160\1276043876.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data_cleaned['latitude'].fillna(mean_coords['latitude'], inplace=True)
C:\Users\κική\AppData\Local\Temp\ipykernel_12160\1276043876.py:8: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves

In [11]:
# Filling columns about numerical data with median and mean
numerical_columns = ['minimum_nights', 'number_of_reviews',
                     'reviews_per_month', 'calculated_host_listings_count',
                     'availability_365']
for col in numerical_columns:
    data_cleaned[col].fillna(data_cleaned[col].median(), inplace=True)

C:\Users\κική\AppData\Local\Temp\ipykernel_12160\100482101.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data_cleaned[col].fillna(data_cleaned[col].median(), inplace=True)
C:\Users\κική\AppData\Local\Temp\ipykernel_12160\100482101.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a 

In [12]:
# Filling columns about numerical data with mode
categorical_columns = ['room_type']
for col in categorical_columns:
    data_cleaned[col].fillna(data_cleaned[col].mode()[0], inplace=True)

C:\Users\κική\AppData\Local\Temp\ipykernel_12160\2510382449.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data_cleaned[col].fillna(data_cleaned[col].mode()[0], inplace=True)


In [13]:
#'price_per_night' - price divided by minimum nights (if minimum nights is 0, set to 1 to avoid division by zero)
data_cleaned['minimum_nights'] = data_cleaned['minimum_nights'].replace(0, 1)
data_cleaned['price_per_night'] = data_cleaned['price'] / data_cleaned['minimum_nights'] 

## Feature engineering

In [14]:
# Applying one-hot encoding
onehot_encoder = OneHotEncoder(sparse_output=False, drop='first') # drop='first' to avoid multicollinearity
encoded_data = pd.DataFrame(onehot_encoder.fit_transform(data_cleaned[categorical_columns]))
encoded_data.columns = onehot_encoder.get_feature_names_out(categorical_columns)

In [15]:
# Dropping original categorical columns and adding encoded columns
#data_fe variable created for feature engineering the cleaned data
data_fe = data_cleaned.drop(columns=categorical_columns)
data_fe = pd.concat([data_fe, encoded_data], axis=1)

In [16]:
non_num_columns = ['neighbourhood']
data_fe = data_fe.drop(columns=non_num_columns,axis=1)
data_fe.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4024 entries, 0 to 4023
Data columns (total 11 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   latitude                        4024 non-null   float64
 1   longitude                       4024 non-null   float64
 2   price                           4024 non-null   float64
 3   minimum_nights                  4024 non-null   int64  
 4   number_of_reviews               4024 non-null   int64  
 5   reviews_per_month               4024 non-null   float64
 6   calculated_host_listings_count  4024 non-null   int64  
 7   availability_365                4024 non-null   int64  
 8   number_of_reviews_ltm           4024 non-null   int64  
 9   price_per_night                 4024 non-null   float64
 10  room_type_Private room          4024 non-null   float64
dtypes: float64(6), int64(5)
memory usage: 345.9 KB


In [17]:
data_fe.isnull().head()

,latitude,longitude,price,minimum_nights,number_of_reviews,reviews_per_month,calculated_host_listings_count,availability_365,number_of_reviews_ltm,price_per_night,room_type_Private room
0,False,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,False


In [18]:
data_fe.head()

,latitude,longitude,price,minimum_nights,number_of_reviews,reviews_per_month,calculated_host_listings_count,availability_365,number_of_reviews_ltm,price_per_night,room_type_Private room
0,40.59683,22.953940,50.0,28,19,0.12,1,363,0,1.785714,0.0
1,40.63855,22.950674,52.0,2,416,2.62,1,356,23,26.000000,0.0
2,40.64763,22.943090,37.0,20,47,0.32,36,84,1,1.850000,0.0
3,40.64008,22.955980,40.0,7,10,0.06,6,337,2,5.714286,0.0
4,40.58628,23.033930,109.0,7,7,0.08,1,266,0,15.571429,0.0


## Train-test split

In [19]:
X = data_fe.drop('price', axis=1)
y = data_fe['price']

In [20]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

## Buliding and training the model

#### Applying Linear Regression

In [21]:
# linear_reg = LinearRegression()
# linear_reg.fit(X_train, y_train)

#### Applying Decision Tree Regression

In [22]:
decision_tree_reg = DecisionTreeRegressor(random_state=101)
decision_tree_reg.fit(X_train, y_train)

DecisionTreeRegressor(random_state=101)

## Evaluate the model 

In [23]:
# y_pred_linear_reg = linear_reg.predict(X_test)

In [24]:
# mse_linear_reg = mean_squared_error(y_test, y_pred_linear_reg)
# r2_linear_reg = r2_score(y_test, y_pred_linear_reg)

# mse_linear_reg, r2_linear_reg
# (0.0035319561452903264, 0.0012006797969080774)

In [25]:
# Predicting on the test set
y_pred_decision_tree = decision_tree_reg.predict(X_test)

In [26]:
mse_decision_tree = mean_squared_error(y_test, y_pred_decision_tree)
r2_decision_tree = r2_score(y_test, y_pred_decision_tree)

mse_decision_tree, r2_decision_tree

(633.5384036144578, 0.7748786446593635)

In [27]:
print(f"Mean Squared Error: {mse_decision_tree:.2f}")
print(f"R^2 Score: {r2_decision_tree:.2f}")

Mean Squared Error: 633.54
R^2 Score: 0.77
